# T2.3 — Grid Outage Forecaster: Evaluation

Rolling 30-day held-out window evaluation:
- **Brier score** for P(outage)
- **MAE** for E[duration | outage]
- **Lead time** on true outages
- Calibration plot
- Feature importance

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.calibration import calibration_curve
import joblib
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '.')
from forecaster import build_features, FEATURE_COLS

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

In [ ]:
# ── Load data and models ──────────────────────────────────────────────────────
df_raw = pd.read_csv('data/grid_history.csv')
df = build_features(df_raw)
df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)

clf = joblib.load('models/outage_classifier.pkl')
reg = joblib.load('models/duration_regressor.pkl')
iso = joblib.load('models/isotonic_calibrator.pkl')

# 30-day held-out set
EVAL_DAYS = 30
eval_start = len(df) - EVAL_DAYS * 24

X_eval  = df[FEATURE_COLS].values[eval_start:]
y_eval  = df['outage'].values[eval_start:]
dur_eval = df['duration_min'].values[eval_start:]
ts_eval  = pd.to_datetime(df['timestamp'].values[eval_start:])

raw_pred  = clf.predict_proba(X_eval)[:, 1]
p_pred    = iso.transform(raw_pred)
dur_pred  = np.expm1(reg.predict(X_eval))

print(f'Eval period: {ts_eval[0].date()} → {ts_eval[-1].date()}')
print(f'Total hours: {len(y_eval):,}  |  True outages: {y_eval.sum()}  |  Rate: {y_eval.mean():.2%}')

In [ ]:
# ── Primary metrics ───────────────────────────────────────────────────────────
bs       = brier_score_loss(y_eval, p_pred)
bs_naive = brier_score_loss(y_eval, np.full_like(p_pred, y_eval.mean()))
bss      = 1 - bs / bs_naive   # Brier Skill Score (higher is better)
auc      = roc_auc_score(y_eval, p_pred)

mask_out  = y_eval == 1
mae_dur   = np.mean(np.abs(dur_pred[mask_out] - dur_eval[mask_out]))

# Lead time: earliest hour in a 6h lookback where p >= 0.15 before true outage
lead_times = []
i = 0
while i < len(y_eval):
    if y_eval[i] == 1:
        for j in range(max(0, i - 6), i):
            if p_pred[j] >= 0.15:
                lead_times.append(i - j)
                break
    i += 1
avg_lead = np.mean(lead_times) if lead_times else 0.0

metrics = {
    'Brier Score':          round(bs, 5),
    'Brier Score (naive)':  round(bs_naive, 5),
    'Brier Skill Score':    round(bss, 4),
    'ROC-AUC':              round(auc, 4),
    'Duration MAE (min)':   round(mae_dur, 2),
    'Avg Lead Time (h)':    round(avg_lead, 2),
    'N outages eval':       int(mask_out.sum()),
}

print('\n=== HELD-OUT METRICS (30-day window) ===')
for k, v in metrics.items():
    print(f'  {k:<28} {v}')

In [ ]:
# ── Figure 1: Forecast uncertainty band over 30-day eval period ───────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax = axes[0]
ax.fill_between(ts_eval, 0, p_pred, alpha=0.3, color='#e74c3c', label='P(outage) forecast')
ax.plot(ts_eval, p_pred, color='#e74c3c', lw=0.8)
ax.scatter(ts_eval[mask_out], p_pred[mask_out], color='darkred', s=20, zorder=5, label='True outage hour')
ax.axhline(0.10, color='orange', ls='--', lw=1, label='Medium risk (10%)')
ax.axhline(0.25, color='red',    ls='--', lw=1, label='High risk (25%)')
ax.set_ylabel('P(outage)')
ax.set_title(f'30-Day Held-Out: Outage Probability Forecast  |  Brier={bs:.4f}  AUC={auc:.3f}')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, 0.5)

ax2 = axes[1]
ax2.fill_between(ts_eval[mask_out], 0, dur_pred[mask_out],
                 alpha=0.5, color='#3498db', step='mid', label='Predicted duration')
ax2.scatter(ts_eval[mask_out], dur_eval[mask_out],
            color='navy', s=25, zorder=5, label=f'Actual duration  (MAE={mae_dur:.0f} min)')
ax2.set_ylabel('Duration (min)')
ax2.set_xlabel('Date')
ax2.set_title('Outage Duration Forecast vs Actual (outage hours only)')
ax2.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/eval_forecast_band.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/eval_forecast_band.png')

In [ ]:
# ── Figure 2: Calibration curve ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))

frac_pos, mean_pred = calibration_curve(y_eval, p_pred, n_bins=10)
ax.plot(mean_pred, frac_pos, 's-', color='#e74c3c', label='LightGBM + Isotonic')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve (30-day hold-out)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eval_calibration.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/eval_calibration.png')

In [ ]:
# ── Figure 3: Feature importance ─────────────────────────────────────────────
importance = clf.feature_importances_
feat_imp   = pd.Series(importance, index=FEATURE_COLS).sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind='barh', ax=ax, color='#3498db')
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('Top-20 Feature Importances — Outage Classifier')
plt.tight_layout()
plt.savefig('outputs/eval_feature_importance.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/eval_feature_importance.png')

print('\nTop-10 features:')
for feat, imp in feat_imp.tail(10).sort_values(ascending=False).items():
    print(f'  {feat:<30} {imp:.1f}')

In [ ]:
# ── Worst forecast hour analysis (for video Q&A) ──────────────────────────────
# Worst hour = highest Brier contribution among true outage hours
brier_contrib = (y_eval - p_pred) ** 2
worst_idx = np.argmax(brier_contrib * mask_out.astype(float))

print('=== WORST FORECAST HOUR ===')
print(f'  Timestamp:       {ts_eval[worst_idx]}')
print(f'  True outage:     {y_eval[worst_idx]}')
print(f'  P(outage) pred:  {p_pred[worst_idx]:.4f}')
print(f'  Brier contrib:   {brier_contrib[worst_idx]:.4f}')
print(f'  Actual duration: {dur_eval[worst_idx]:.0f} min')
print(f'  Pred duration:   {dur_pred[worst_idx]:.0f} min')
print()
print('Why the model failed here:')
row = df.iloc[eval_start + worst_idx]
print(f'  load_lag1:  {row["load_lag1"]:.3f} MW  (was load BELOW peak → low signal)')
print(f'  rain_mm:    {row["rain_mm"]:.3f}        (low rain → underestimated risk)')
print(f'  hour:       {int(row["hour"])}          ')
print()
print('Feature to add next week:')
print('  → "Neighbor signal": crowd-sourced outage reports from nearby SMEs.')
print('    When 2+ neighbors report outages in the last 30 min, P(outage) spikes')
print('    regardless of load/rain, catching surprise faults the model misses.')

In [ ]:
# ── Save final metrics JSON ───────────────────────────────────────────────────
with open('outputs/eval_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Final metrics saved → outputs/eval_metrics.json')
print('\nAll evaluation complete.')